In [0]:
dbutils.library.restartPython()

In [0]:
import sys

sys.path.append("/Workspace/Users/sjaddya98@gmail.com/energy-lakehouse/src")

In [0]:
import uuid
from datetime import UTC, datetime

from energy_lakehouse.bronze import bronze_weather_rows
from energy_lakehouse.dwd import (
    BASE_URL,
    PARAMETERS,
    STATION_IDS,
    check_stations_alive,
    fetch_recent,
    fetch_station_list,
    parse_product,
    read_product,
    recent_data_path,
)
from energy_lakehouse.http_client import make_client

TABLE = "workspace.bronze.dwd_raw"

load_id = str(uuid.uuid4())
fetched_at = datetime.now(UTC)
client = make_client(BASE_URL, timeout_seconds=60.0)

print(load_id, fetched_at)

In [0]:
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

WEATHER_SCHEMA = StructType(
    [
        StructField("station_id", StringType(), False),
        StructField("timestamp_utc", TimestampType(), False),
        StructField("parameter", StringType(), False),
        StructField("value", DoubleType(), True),
        StructField("quality", IntegerType(), True),
        StructField("source_file", StringType(), False),
        StructField("load_id", StringType(), False),
        StructField("fetched_at", TimestampType(), False),
    ]
)

if not spark.catalog.tableExists(TABLE):
    spark.createDataFrame([], WEATHER_SCHEMA).write.saveAsTable(TABLE)
    print("created", TABLE)
else:
    print("exists", TABLE)

In [0]:
today = fetched_at.date()

for parameter in PARAMETERS:
    stations = fetch_station_list(client, parameter)
    check_stations_alive(stations, STATION_IDS, today)
    print(f"{parameter.folder}: {len(stations)} stations listed, all 8 configured ones current")

In [0]:
for parameter in PARAMETERS:
    for station_id in STATION_IDS:
        archive = fetch_recent(client, parameter, station_id)
        readings = parse_product(read_product(archive), parameter)
        rows = bronze_weather_rows(
            readings,
            recent_data_path(parameter, station_id),
            load_id,
            fetched_at,
        )

        if not rows:
            print(f"{parameter.folder} {station_id}: nothing to load")
            continue

        df = spark.createDataFrame(rows, WEATHER_SCHEMA)
        df.createOrReplaceTempView("incoming_weather")
        spark.sql(f"""
            MERGE INTO {TABLE} AS target
            USING incoming_weather AS source
               ON target.station_id = source.station_id
              AND target.timestamp_utc = source.timestamp_utc
              AND target.parameter = source.parameter
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        print(f"{parameter.folder} {station_id}: {len(rows)} rows merged")

In [0]:
%sql
SELECT parameter,
       count(*)                   AS n_rows,
       count(value)               AS n_values,
       count(DISTINCT station_id) AS n_stations,
       min(timestamp_utc)         AS first_hour,
       max(timestamp_utc)         AS last_hour,
       count(DISTINCT load_id)    AS n_loads
FROM workspace.bronze.dwd_raw
GROUP BY parameter
ORDER BY parameter

In [0]:
%sql
SELECT count(*) AS duplicate_keys
FROM (
  SELECT station_id, timestamp_utc, parameter
  FROM workspace.bronze.dwd_raw
  GROUP BY station_id, timestamp_utc, parameter
  HAVING count(*) > 1
)